<a href="https://colab.research.google.com/github/salphonseds/llm-from-scratch/blob/main/notebooks/12_Integration_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1 — Setup: GPT-2 small, WikiText-2 general eval, canonical harness
# ==========================================================================
# NB12: Integration — the full pipeline (Day 21)
# pretrained -> SFT -> merge -> quantize -> serve, one yardstick throughout.
# Measurement rule: every PPL in this notebook comes from the SAME harness
# as notebooks 09/10/11 (tuple return, non-overlapping 1024-token windows),
# or the cross-notebook ledger is meaningless.

import torch, torch.nn as nn, torch.nn.functional as F
import math, time, gc, random, copy
import numpy as np
from collections import Counter
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from datasets import load_dataset

device = "cuda"
assert torch.cuda.is_available(), "T4 not attached"
print(f"GPU: {torch.cuda.get_device_name(0)}")

torch.manual_seed(42); random.seed(42); np.random.seed(42)

model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
n_params = sum(p.numel() for p in model.parameters())
print(f"GPT-2 small: {n_params:,} parameters")
assert n_params == 124_439_808, "not the curriculum checkpoint"

wikitext = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
general_text = "\n\n".join(t for t in wikitext["text"] if t.strip())
general_ids = tokenizer(general_text, return_tensors="pt").input_ids[0]
print(f"WikiText-2 test: {len(general_ids):,} tokens")

@torch.no_grad()
def perplexity(model, ids, ctx_len=1024, max_windows=None):
    """Mean NLL over non-overlapping windows -> exp. Simple + fast; slightly
    pessimistic vs. sliding-window eval (early tokens in each window get
    little context), but the bias is identical before/after fine-tuning,
    which is all we need for a forgetting delta."""
    model.eval()
    nll_sum, tok_count = 0.0, 0
    windows = range(0, len(ids) - 1, ctx_len)
    if max_windows is not None:
        windows = list(windows)[:max_windows]
    for start in windows:
        chunk = ids[start : start + ctx_len + 1].to(device)
        if len(chunk) < 2:
            break
        x, y = chunk[:-1].unsqueeze(0), chunk[1:].unsqueeze(0)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = model(x).logits
        loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), y.view(-1))
        n = y.numel()
        nll_sum += loss.item() * n
        tok_count += n
    return math.exp(nll_sum / tok_count), tok_count

t0 = time.time()
ppl_general, n_tok = perplexity(model, general_ids)
print(f"Pristine general PPL: {ppl_general:.2f}  ({n_tok:,} tokens, {time.time()-t0:.1f}s)")
assert abs(ppl_general - 29.03) < 0.02, "harness or checkpoint drifted — STOP"
print("✓ Acceptance gate: 29.03 — the unbroken NB09/10/11 thread continues")

GPU: Tesla T4


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (286177 > 1024). Running this sequence through the model will result in indexing errors


WikiText-2 test: 286,177 tokens
Pristine general PPL: 29.03  (286,176 tokens, 11.3s)
✓ Acceptance gate: 29.03 — the unbroken NB09/10/11 thread continues


In [ ]:
# Cell 2 — Dolly-15k: load, inspect, and carve the held-out split
#
# Human-written instruction data (no model-generated responses) — the clean
# experiment for observing what SFT does. Split is carved FIRST, seeded,
# before any formatting or training code exists to contaminate it.

dolly = load_dataset("databricks/databricks-dolly-15k", split="train")
print(f"Dolly-15k: {len(dolly):,} examples")
print(f"Fields: {dolly.column_names}\n")

# ---- category census ----
from collections import Counter
cats = Counter(dolly["category"])
for c, n in cats.most_common():
    print(f"  {c:>24}: {n:>5}  ({100*n/len(dolly):.1f}%)")

# ---- context field: how many examples actually use it? ----
has_ctx = sum(1 for c in dolly["context"] if c.strip())
print(f"\nExamples with non-empty context: {has_ctx:,} ({100*has_ctx/len(dolly):.1f}%)")

# ---- token-length distributions (tokenize once, keep the counts) ----
# Full-corpus tokenization here is ~30-60s; we keep only lengths, not ids.
instr_lens, resp_lens = [], []
for ex in dolly:
    prompt_text = ex["instruction"] + (("\n" + ex["context"]) if ex["context"].strip() else "")
    instr_lens.append(len(tokenizer(prompt_text).input_ids))
    resp_lens.append(len(tokenizer(ex["response"]).input_ids))
instr_lens = np.array(instr_lens); resp_lens = np.array(resp_lens)

def stats(name, a):
    print(f"  {name:>10}: median {np.median(a):>5.0f} | mean {a.mean():>6.1f} | "
          f"p90 {np.percentile(a,90):>5.0f} | p99 {np.percentile(a,99):>6.0f} | max {a.max():>6}")
print("\nToken lengths (prompt = instruction [+ context]):")
stats("prompt", instr_lens)
stats("response", resp_lens)
total = instr_lens + resp_lens
print(f"\nExamples with prompt+response > 900 tokens: {(total > 900).sum():,} "
      f"({100*(total>900).mean():.1f}%)  <- template overhead budget: ~24 tokens")

# ---- held-out split: seeded, carved before anything else ----
idx = np.arange(len(dolly)); rng = np.random.default_rng(42); rng.shuffle(idx)
N_HELDOUT = 500
heldout_idx, train_idx = idx[:N_HELDOUT].tolist(), idx[N_HELDOUT:].tolist()
dolly_heldout = dolly.select(heldout_idx)
dolly_train   = dolly.select(train_idx)
print(f"\nSplit: {len(dolly_train):,} train | {len(dolly_heldout)} held-out (seeded, deterministic)")
print(f"First 5 held-out indices: {heldout_idx[:5]}  <- record these; they gate reproducibility")

README.md:   0%|          | 0.00/8.20k [00:00<?, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

Dolly-15k: 15,011 examples
Fields: ['instruction', 'context', 'response', 'category']

                   open_qa:  3742  (24.9%)
                general_qa:  2191  (14.6%)
            classification:  2136  (14.2%)
                 closed_qa:  1773  (11.8%)
             brainstorming:  1766  (11.8%)
    information_extraction:  1506  (10.0%)
             summarization:  1188  (7.9%)
          creative_writing:   709  (4.7%)

Examples with non-empty context: 4,467 (29.8%)

Token lengths (prompt = instruction [+ context]):
      prompt: median    16 | mean   93.7 | p90   258 | p99    868 | max   5579
    response: median    44 | mean   79.0 | p90   173 | p99    560 | max   5209

Examples with prompt+response > 900 tokens: 244 (1.6%)  <- template overhead budget: ~24 tokens

Split: 14,511 train | 500 held-out (seeded, deterministic)
First 5 held-out indices: [13384, 10119, 423, 10112, 12021]  <- record these; they gate reproducibility


In [ ]:
# Cell 3 — NB11 yardstick transplant 1/2: template, exact boundary, collate
#
# Functions verbatim from NB11 Cells 3-4. The template is the conditioning
# signal — inference must reproduce it byte-exactly. Prompt and response
# tokenized SEPARATELY so the boundary is exact by construction (BPE is
# not concat-invariant). NB11's demo-loss and signal-ratio blocks dropped:
# that experiment is settled; this notebook only inherits the yardstick.

TEMPLATE_NO_CTX = "### Instruction:\n{instruction}\n\n### Response:\n"
TEMPLATE_CTX    = "### Instruction:\n{instruction}\n\n### Context:\n{context}\n\n### Response:\n"
EOS_ID = tokenizer.eos_token_id  # 50256, <|endoftext|>
MAX_LEN = 512                    # NB09/10/11 training shape; longer examples DROPPED

def format_prompt(ex):
    if ex["context"].strip():
        return TEMPLATE_CTX.format(instruction=ex["instruction"], context=ex["context"])
    return TEMPLATE_NO_CTX.format(instruction=ex["instruction"])

def encode_example(ex):
    """-> (input_ids list, boundary int) or None if over budget.
    boundary = index of first RESPONSE token; ids[boundary:] includes EOS."""
    p_ids = tokenizer(format_prompt(ex)).input_ids
    r_ids = tokenizer(ex["response"]).input_ids + [EOS_ID]
    ids = p_ids + r_ids
    if len(ids) > MAX_LEN:
        return None
    return ids, len(p_ids)

# ---- verification on two hand-checked examples (one per template) ----
ex_no_ctx = next(e for e in dolly_train if not e["context"].strip())
ex_ctx    = next(e for e in dolly_train if e["context"].strip())
for tag, ex in [("no-context", ex_no_ctx), ("with-context", ex_ctx)]:
    out = encode_example(ex)
    assert out is not None, f"verification example ({tag}) over budget — pick another"
    ids, b = out
    assert ids[-1] == EOS_ID
    assert tokenizer.decode(ids[:b]).endswith("### Response:\n")
    print(f"[{tag}] total {len(ids)} tokens | boundary {b} | "
          f"first response tokens: {tokenizer.decode(ids[b:b+8])!r}")

# ---- encode both pools, count the drops ----
def encode_pool(pool):
    kept, dropped_cats = [], Counter()
    for ex in pool:
        out = encode_example(ex)
        if out is None:
            dropped_cats[ex["category"]] += 1
        else:
            kept.append(out)
    return kept, dropped_cats

train_encoded, train_drops = encode_pool(dolly_train)
held_encoded,  held_drops  = encode_pool(dolly_heldout)

n_drop = sum(train_drops.values())
print(f"\nTrain pool: {len(train_encoded):,} kept | {n_drop:,} dropped ({100*n_drop/len(dolly_train):.1f}%)")
print("Drops by category (train):")
for c, n in train_drops.most_common():
    print(f"  {c:>24}: {n:>4}  ({100*n/cats[c]:.1f}% of category)")
print(f"\nHeld-out: {len(held_encoded)} kept | {sum(held_drops.values())} dropped")
resp_tok_total = sum(len(ids) - b for ids, b in train_encoded)
print(f"Trainable response tokens in kept train pool: {resp_tok_total:,}")

# ---- collate: NB11 Cell 4 verbatim ----
PAD_ID = EOS_ID  # GPT-2 has no pad token; attention_mask + label -100 neutralize it

def collate(batch, mask_prompt=True):
    """batch: list of (ids, boundary). -> input_ids, attention_mask, labels."""
    maxlen = max(len(ids) for ids, _ in batch)
    B = len(batch)
    input_ids = torch.full((B, maxlen), PAD_ID, dtype=torch.long)
    attn      = torch.zeros((B, maxlen), dtype=torch.long)
    labels    = torch.full((B, maxlen), -100, dtype=torch.long)
    for i, (ids, b) in enumerate(batch):
        L = len(ids)
        t = torch.tensor(ids, dtype=torch.long)
        input_ids[i, :L] = t
        attn[i, :L] = 1
        labels[i, :L] = t                      # start from full copy...
        if mask_prompt:
            labels[i, :b] = -100               # ...then silence the prompt
    return input_ids, attn, labels

# ---- collate verification: the mask is where we claim ----
ids, b = train_encoded[0]
x, a, y = collate([train_encoded[0]])
n_loss = (y != -100).sum().item()
assert n_loss == len(ids) - b, "unmasked count != response length"
assert y[0, b].item() == ids[b] and y[0, b-1].item() == -100
print(f"\nCollate check: example 0, {len(ids)} tokens, boundary {b} -> {n_loss} loss-carrying labels ✓")

# ---- NB12-specific gates: NB11's measured constants, now inherited ----
assert len(train_encoded) == 13_659, "train pool drifted from NB11"
assert len(held_encoded)  == 463,    "held-out pool drifted from NB11"
assert resp_tok_total     == 918_503, "response token count drifted from NB11"
print("✓ Pools match NB11 exactly: 13,659 train | 463 held-out | 918,503 response tokens")

[no-context] total 211 tokens | boundary 18 | first response tokens: 'Cars are vehicles that allow you to'
[with-context] total 106 tokens | boundary 83 | first response tokens: 'The founding members of id Software were John'

Train pool: 13,659 kept | 852 dropped (5.9%)
Drops by category (train):
    information_extraction:  272  (18.1% of category)
             summarization:  258  (21.7% of category)
                 closed_qa:  178  (10.0% of category)
          creative_writing:   62  (8.7% of category)
                general_qa:   42  (1.9% of category)
                   open_qa:   17  (0.5% of category)
             brainstorming:   12  (0.7% of category)
            classification:   11  (0.5% of category)

Held-out: 463 kept | 37 dropped
Trainable response tokens in kept train pool: 918,503

Collate check: example 0, 211 tokens, boundary 18 -> 193 loss-carrying labels ✓
✓ Pools match NB11 exactly: 13,659 train | 463 held-out | 918,503 response tokens


In [ ]:
# Cell 4 — NB11 yardstick transplant 2/2: response PPL + frozen suite
#
# response_ppl: NB11 Cell 5a verbatim. Suite: NB11 5b with the 5c
# attention-mask patch folded in (the frozen, canonical form).
# Pristine measurements here = the quality half of the ledger's Stage 1 row.

@torch.no_grad()
def response_ppl(model, encoded, batch_size=4):
    model.eval()
    nll_sum, tok_count = 0.0, 0
    for i in range(0, len(encoded), batch_size):
        x, a, y = collate(encoded[i : i + batch_size], mask_prompt=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            out = model(x.to(device), attention_mask=a.to(device), labels=y.to(device))
        n = (y != -100).sum().item()
        nll_sum += out.loss.item() * n
        tok_count += n
    return math.exp(nll_sum / tok_count), tok_count

def make_prompt(instruction, context=""):
    ex = {"instruction": instruction, "context": context}
    return format_prompt(ex)

SUITE_A = [
    ("one_word",   "Answer in exactly one word: what is the capital of France?",
     lambda t: len(t.strip().rstrip(".!").split()) == 1),
    ("three_lines","List exactly three animals, one per line.",
     lambda t: len([l for l in t.strip().splitlines() if l.strip()]) == 3),
    ("uppercase",  "Respond only in uppercase letters: name a color.",
     lambda t: t.strip() != "" and t.upper() == t),
    ("yes_no",     "Answer only 'yes' or 'no': is the sun a star?",
     lambda t: t.strip().lower().rstrip(".!") in ("yes", "no")),
    ("count_5",    "Count from 1 to 5, separated by commas.",
     lambda t: all(str(d) in t for d in range(1, 6))),
    ("ctx_extract","What year is mentioned in the context? Answer with just the year.",
     lambda t: t.strip().rstrip(".") == "1969",
     "The Apollo 11 mission landed the first humans on the Moon in 1969."),
]
SUITE_B = [
    ("explain",  "Explain why the sky is blue in two sentences."),
    ("brainstm", "Give me three ideas for a birthday gift for a chess enthusiast."),
    ("factual",  "Who wrote the novel Moby-Dick, and in what year was it published?"),
    ("refuse",   "Translate this sentence into French: 'The library opens at nine.'"),
]

def _generate(model, ids):
    return model.generate(ids, attention_mask=torch.ones_like(ids),
                          max_new_tokens=80, do_sample=False,
                          eos_token_id=EOS_ID, pad_token_id=EOS_ID)

@torch.no_grad()
def run_suite(model, verbose=True):
    model.eval()
    results = {}
    for item in SUITE_A + SUITE_B:
        name, instr = item[0], item[1]
        ctx = item[3] if len(item) > 3 else ""
        ids = tokenizer(make_prompt(instr, ctx), return_tensors="pt").input_ids.to(device)
        out = _generate(model, ids)
        gen = out[0, ids.shape[1]:]
        stopped = EOS_ID in gen.tolist()
        text = tokenizer.decode(gen, skip_special_tokens=True)
        passed = item[2](text) if len(item) > 2 and callable(item[2]) else None
        results[name] = (text, passed, stopped)
        if verbose:
            tag = " " if passed is None else ("PASS" if passed else "FAIL")
            print(f"[{name:>11}] {tag} | stopped: {stopped} | {text[:90]!r}")
    n_pass = sum(1 for _, p, _ in results.values() if p)
    n_stop = sum(1 for _, _, s in results.values() if s)
    print(f"\nTier A: {n_pass}/6 checks passed | EOS self-stop: {n_stop}/10 prompts")
    return results

# ---- pristine measurements: quality half of the Stage 1 ledger row ----
t0 = time.time()
ppl_resp_pristine, n_resp = response_ppl(model, held_encoded)
print(f"Pristine held-out response PPL: {ppl_resp_pristine:.2f}  "
      f"({n_resp:,} response tokens, {time.time()-t0:.0f}s)")
assert abs(ppl_resp_pristine - 20.96) < 0.02, "response harness drifted from NB11"
assert n_resp == 31_793, "held-out response token count drifted from NB11"
print("✓ Canonical harness #2: 20.96 / 31,793 — NB11 constants inherited\n")

print("=== PRISTINE GPT-2 — behavioral suite (Stage 1 reference) ===\n")
suite_pristine = run_suite(model)

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Pristine held-out response PPL: 20.96  (31,793 response tokens, 5s)
✓ Canonical harness #2: 20.96 / 31,793 — NB11 constants inherited

=== PRISTINE GPT-2 — behavioral suite (Stage 1 reference) ===

[   one_word] FAIL | stopped: False | '\nAnswer in exactly one word: what is the capital of France?\n\n### Response:\n\nAnswer in exac'
[three_lines] FAIL | stopped: False | '\nThe following is a list of all the animals in the list.\n\n# List of animals in the list.\n\n'
[  uppercase] FAIL | stopped: False | '\nRespond only in uppercase letters: name a color.\n\n### Response:\n\nRespond only in uppercas'
[     yes_no] FAIL | stopped: False | "\nAnswer only 'yes' or 'no': is the sun a star?\n\n### Response:\n\nAnswer only 'yes' or 'no': "
[    count_5] FAIL | stopped: False | '\nCount from 1 to 5, separated by commas.\n\n### Response:\n\nCount from 1 to 5, separated by c'
[ctx_extract] FAIL | stopped: False | '\nThe Apollo 11 mission landed the first humans on the Moon in 1969.\n\n### Respons

In [ ]:
# Cell 5 — The pipeline contract: stages, deployment ledger, cost harness
# ==========================================================================
#   Stage 1  pretrained    — pristine GPT-2, fp32 and fp16 reference rows
#   Stage 2  fine-tune     — LoRA SFT (NB11 winner: r=8, 3e-4, masked)
#   Stage 3  merge         — fold BA into W (NB10 machinery), adapters gone
#   Stage 4  quantize      — NF4 on the MERGED model (serving, not training)
#   Stage 5  serve         — KV-cache × dtype × length latency matrix
#
# Ledger: 3 quality axes (canonical harnesses 1-3) x 3 cost axes (below).
# Cost harness rule: EXACT-LENGTH generation (min=max=128) — tokens/sec must
# not be confounded by learned EOS stopping. MB = 1e6 bytes throughout.

BENCH_PROMPT = make_prompt("Explain how a lighthouse works.")
N_NEW = 128

def gen_speed(model, n_new=N_NEW, n_runs=3, use_cache=True):
    """Median tokens/sec, greedy, batch 1, forced n_new tokens."""
    ids = tokenizer(BENCH_PROMPT, return_tensors="pt").input_ids.to(device)
    mask = torch.ones_like(ids)
    def _run():
        torch.cuda.synchronize(); t0 = time.time()
        model.generate(ids, attention_mask=mask,
                       max_new_tokens=n_new, min_new_tokens=n_new,
                       do_sample=False, eos_token_id=EOS_ID, pad_token_id=EOS_ID,
                       use_cache=use_cache)
        torch.cuda.synchronize()
        return n_new / (time.time() - t0)
    _run()                                              # warmup, untimed
    return float(np.median([_run() for _ in range(n_runs)]))

def weights_mb(model):
    return sum(p.numel() * p.element_size() for p in model.parameters()) / 1e6

def peak_infer_mb(model):
    """Peak CUDA memory during one benchmark generation."""
    gc.collect(); torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    gen_speed(model, n_runs=1)
    return torch.cuda.max_memory_allocated() / 1e6

LEDGER = {}
COLS = ["general", "resp", "tierA", "stop", "weights_mb", "peak_mb", "tok_s"]

def record(stage, **kw):
    LEDGER[stage] = {c: kw.get(c) for c in COLS}
    print(f"[ledger] {stage}: " + " | ".join(
        f"{c}={v:.2f}" if isinstance(v, float) else f"{c}={v}"
        for c, v in LEDGER[stage].items() if v is not None))

def show_ledger():
    hdr = f"{'stage':>16} |" + "".join(f"{c:>12} |" for c in COLS)
    print(hdr); print("-" * len(hdr))
    for s, row in LEDGER.items():
        line = f"{s:>16} |"
        for c in COLS:
            v = row[c]
            line += f"{'-':>12} |" if v is None else (
                f"{v:>12.2f} |" if isinstance(v, float) else f"{str(v):>12} |")
        print(line)

n_bench_tok = len(tokenizer(BENCH_PROMPT).input_ids)
print(f"Cost harness defined. Benchmark prompt: {n_bench_tok} tokens, "
      f"forced {N_NEW} new tokens, greedy, batch 1, median of 3.")
print(f"Ledger schema: {len(COLS)} columns x 5+ stage rows. Contract on record.")

Cost harness defined. Benchmark prompt: 17 tokens, forced 128 new tokens, greedy, batch 1, median of 3.
Ledger schema: 7 columns x 5+ stage rows. Contract on record.


In [ ]:
# Cell 6 — Stage 1: pristine reference rows, fp32 and fp16
#
# fp32 quality axes reused from Cells 1/4 (already measured, same model).
# .half() is deployment transformation #1: all axes re-measured, nothing
# assumed. Suite divergence counted — greedy argmax chains can split at
# a single precision-flipped near-tie.

def suite_scores(results):
    n_pass = sum(1 for _, p, _ in results.values() if p)
    n_stop = sum(1 for _, _, s in results.values() if s)
    return n_pass, n_stop

# ---- fp32 row: quality known, cost measured now ----
w32 = weights_mb(model)
print(f"fp32 weights: {w32:.2f} MB")
tok_s_32 = gen_speed(model)
peak_32  = peak_infer_mb(model)
nA, nS = suite_scores(suite_pristine)
record("pristine_fp32",
       general=29.03, resp=float(f"{ppl_resp_pristine:.2f}"),
       tierA=f"{nA}/6", stop=f"{nS}/10",
       weights_mb=w32, peak_mb=peak_32, tok_s=tok_s_32)

# ---- deployment transformation #1: fp16 storage ----
model = model.half()
gc.collect(); torch.cuda.empty_cache()
w16 = weights_mb(model)
print(f"\nfp16 weights: {w16:.2f} MB  (ratio {w32/w16:.3f}x)")

ppl_gen_16, _  = perplexity(model, general_ids)
ppl_resp_16, _ = response_ppl(model, held_encoded)
print(f"fp16 quality: general {ppl_gen_16:.2f} (fp32: 29.03) | "
      f"resp {ppl_resp_16:.2f} (fp32: {ppl_resp_pristine:.2f})")

print("\n=== fp16 behavioral suite ===\n")
suite_16 = run_suite(model)
n_diverged = sum(1 for k in suite_pristine
                 if suite_16[k][0] != suite_pristine[k][0])
print(f"\nGenerations diverged from fp32: {n_diverged}/10")

tok_s_16 = gen_speed(model)
peak_16  = peak_infer_mb(model)
nA, nS = suite_scores(suite_16)
record("pristine_fp16",
       general=float(f"{ppl_gen_16:.2f}"), resp=float(f"{ppl_resp_16:.2f}"),
       tierA=f"{nA}/6", stop=f"{nS}/10",
       weights_mb=w16, peak_mb=peak_16, tok_s=tok_s_16)

print(f"\nSpeed: fp32 {tok_s_32:.1f} tok/s -> fp16 {tok_s_16:.1f} tok/s "
      f"({tok_s_16/tok_s_32:.2f}x)")
show_ledger()

# ---- park nothing: Cell 7 reloads pristine fp32 for training ----
del model; gc.collect(); torch.cuda.empty_cache()
print("\nfp16 model released. Cell 7 reloads pristine fp32 for Stage 2 training.")

fp32 weights: 497.76 MB
[ledger] pristine_fp32: general=29.03 | resp=20.96 | tierA=0/6 | stop=0/10 | weights_mb=497.76 | peak_mb=520.42 | tok_s=87.69

fp16 weights: 248.88 MB  (ratio 2.000x)
fp16 quality: general 29.03 (fp32: 29.03) | resp 20.96 (fp32: 20.96)

=== fp16 behavioral suite ===

[   one_word] FAIL | stopped: False | '\nAnswer in exactly one word: what is the capital of France?\n\n### Response:\n\nAnswer in exac'
[three_lines] FAIL | stopped: False | '\nThe following is a list of all the animals in the list.\n\n# List of animals in the list.\n\n'
[  uppercase] FAIL | stopped: False | '\nRespond only in uppercase letters: name a color.\n\n### Response:\n\nRespond only in uppercas'
[     yes_no] FAIL | stopped: False | "\nAnswer only 'yes' or 'no': is the sun a star?\n\n### Response:\n\nAnswer only 'yes' or 'no': "
[    count_5] FAIL | stopped: False | '\nCount from 1 to 5, separated by commas.\n\n### Response:\n\nCount from 1 to 5, separated by c'
[ctx_extract] FAIL | stopped

In [ ]:
# Cell 7 — Stage 2 machinery: reload, NB11 recipe, NB10 LoRA, unit tests
#
# sft_train: NB11 Cell 6 verbatim (committed form: optimizer filters on
# requires_grad — no-op for full FT, load-bearing for LoRA).
# LoRA/LoRAConv1D + 3 unit tests: NB10 Cell 5 via NB11 Cell 8, verbatim.
# This cell defines and probes; Cell 8 mutates and trains.

model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
ppl_check, _ = perplexity(model, general_ids)
assert abs(ppl_check - 29.03) < 0.02, "pristine reload failed"
print(f"Pristine reloaded: general PPL {ppl_check:.2f} ✓\n")

# ==== NB11 Cell 6, verbatim: the SFT recipe ====
BATCH, STEPS, WARMUP, LR_PEAK, CLIP = 4, 300, 20, 3e-5, 1.0

def lr_at(step):
    if step < WARMUP:
        return LR_PEAK * (step + 1) / WARMUP
    p = (step - WARMUP) / (STEPS - WARMUP)
    return LR_PEAK * 0.5 * (1 + math.cos(math.pi * p))

def sft_train(model, encoded, mask_prompt, tag, ckpt_every=100):
    """NB09 recipe on instruction data. Returns (loss curve, resp-PPL checkpoints)."""
    order = list(range(len(encoded)))
    random.Random(42).shuffle(order)                    # same data order as NB11
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                                  lr=LR_PEAK, weight_decay=0.01)
    scaler = torch.amp.GradScaler("cuda")
    losses, ckpts = [], {0: response_ppl(model, held_encoded)[0]}
    model.train(); t0 = time.time()
    for step in range(STEPS):
        batch = [encoded[order[(step * BATCH + j) % len(order)]] for j in range(BATCH)]
        x, a, y = collate(batch, mask_prompt=mask_prompt)
        for g in optimizer.param_groups: g["lr"] = lr_at(step)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            out = model(x.to(device), attention_mask=a.to(device), labels=y.to(device))
        scaler.scale(out.loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], CLIP)
        scaler.step(optimizer); scaler.update()
        losses.append(out.loss.item())
        if (step + 1) % ckpt_every == 0:
            ckpts[step + 1] = response_ppl(model, held_encoded)[0]
            model.train()
            print(f"[{tag}] step {step+1:>3} | loss {np.mean(losses[-50:]):.3f} | "
                  f"held-out resp PPL {ckpts[step+1]:.2f} | {time.time()-t0:.0f}s")
    del optimizer, scaler; gc.collect(); torch.cuda.empty_cache()
    return losses, ckpts

# ==== NB10 Cell 5 via NB11 Cell 8, verbatim: LoRA + unit tests ====
from transformers.pytorch_utils import Conv1D

class LoRA(nn.Module):
    """Rank-r additive update for a linear map d_in -> d_out."""
    def __init__(self, d_in, d_out, r, alpha):
        super().__init__()
        self.A = nn.Parameter(torch.randn(r, d_in) * 0.02)   # random: keeps B's grad alive
        self.B = nn.Parameter(torch.zeros(d_out, r))          # zero:   BA=0 -> identity at init
        self.scale = alpha / r
    def forward(self, x):
        return self.scale * ((x @ self.A.T) @ self.B.T)

class LoRAConv1D(nn.Module):
    """Wraps a GPT-2 Conv1D. Adds independent LoRA branches on given
    output-column slices (so fused c_attn gets separate Q/K/V adapters)."""
    def __init__(self, base, slices, r, alpha):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad_(False)                           # structural freeze
        d_in = base.weight.shape[0]                           # Conv1D: (d_in, d_out)
        self.slices = slices
        self.adapters = nn.ModuleList(
            [LoRA(d_in, e - s, r, alpha) for s, e in slices])
    def forward(self, x):
        out = self.base(x)
        for (s, e), ad in zip(self.slices, self.adapters):
            out[..., s:e] = out[..., s:e] + ad(x)
        return out

torch.manual_seed(42)
probe_base = copy.deepcopy(model.transformer.h[0].attn.c_attn).float().cpu()
wrapped = LoRAConv1D(probe_base, slices=[(0, 768), (768, 1536), (1536, 2304)],
                     r=8, alpha=16)
x = torch.randn(2, 5, 768)
with torch.no_grad():
    diff = (wrapped(x) - probe_base(x)).abs().max()
print(f"Test 1 — identity at init: max |wrapped - base| = {diff.item()}")
assert diff.item() == 0.0, "LoRA at init must be exactly the base map"
out = wrapped(x); out.sum().backward()
print(f"Test 2 — base weight grad:  {probe_base.weight.grad}")
print(f"         B grad norm:       {wrapped.adapters[0].B.grad.norm().item():.4f}  (expect > 0)")
print(f"         A grad norm:       {wrapped.adapters[0].A.grad.norm().item():.4f}  (expect exactly 0 — B=0 blocks it)")
n_probe = sum(p.numel() for p in wrapped.parameters() if p.requires_grad)
print(f"Test 3 — trainable in wrapped c_attn: {n_probe:,}  (expect 36,864)")
for _name in ["probe_base", "wrapped", "x", "out", "diff"]:
    if _name in globals():
        del globals()[_name]
gc.collect()
print("\n✓ Machinery defined and probed. Cell 8 injects, gates, and trains.")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Pristine reloaded: general PPL 29.03 ✓

Test 1 — identity at init: max |wrapped - base| = 0.0
Test 2 — base weight grad:  None
         B grad norm:       159.8717  (expect > 0)
         A grad norm:       0.0000  (expect exactly 0 — B=0 blocks it)
Test 3 — trainable in wrapped c_attn: 36,864  (expect 36,864)

✓ Machinery defined and probed. Cell 8 injects, gates, and trains.


In [ ]:
# Cell 8 — Stage 2: LoRA SFT (NB11 winner re-run: r=8, 3e-4, masked)
#
# Injection order matters: GLOBAL freeze first (the NB10 bug), then seed,
# then wrap. Identity through 48 zero-init adapters must read pristine on
# both axes before a single training step is trusted.

R, ALPHA = 8, 16
for p in model.parameters():
    p.requires_grad_(False)          # GLOBAL freeze — embeddings/LNs/MLPs (the NB10 bug)
torch.manual_seed(42)                # reproducible adapter init
for block in model.transformer.h:
    block.attn.c_attn = LoRAConv1D(
        block.attn.c_attn, slices=[(0, 768), (768, 1536), (1536, 2304)],
        r=R, alpha=ALPHA).to(device)
    block.attn.c_proj = LoRAConv1D(
        block.attn.c_proj, slices=[(0, 768)],
        r=R, alpha=ALPHA).to(device)
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f"trainable: {n_train:,} / {n_total:,} = {100*n_train/n_total:.3f}%  "
      f"(expect 589,824 / 0.472%)")
assert n_train == 589_824, "adapter parameter count drifted from NB10/11"

# Identity through 48 zero-init adapters, both quality axes
ppl_g, _ = perplexity(model, general_ids)
ppl_r, _ = response_ppl(model, held_encoded)
print(f"with adapters — general PPL {ppl_g:.2f} (expect 29.03) | "
      f"held-out resp PPL {ppl_r:.2f} (expect 20.96)")
assert abs(ppl_g - 29.03) < 0.02 and abs(ppl_r - ppl_resp_pristine) < 0.02
print("✓ 48 adapters installed, base frozen, identity verified at model scale\n")

# ==== Train at the NB10/11 winning adapter lr ====
LR_PEAK = 3e-4
losses_l, ckpts_l = sft_train(model, train_encoded, mask_prompt=True, tag="lora")
LR_PEAK = 3e-5

ppl_resp_l, _ = response_ppl(model, held_encoded)
ppl_gen_l, _  = perplexity(model, general_ids)
print(f"\nLoRA SFT: held-out resp PPL {ppl_resp_pristine:.2f} -> {ppl_resp_l:.2f} "
      f"(NB11: 14.27) | general PPL 29.03 -> {ppl_gen_l:.2f} (NB11: 29.06)")

print("\n=== Behavioral suite, LoRA SFT (compare NB11 byte-for-byte) ===\n")
suite_lora = run_suite(model)

# ---- Stage 2 ledger row: fp32 base + attached adapters ----
nA, nS = suite_scores(suite_lora)
w_l  = weights_mb(model)
tok_s_l = gen_speed(model)
peak_l  = peak_infer_mb(model)
record("lora_sft_fp32",
       general=float(f"{ppl_gen_l:.2f}"), resp=float(f"{ppl_resp_l:.2f}"),
       tierA=f"{nA}/6", stop=f"{nS}/10",
       weights_mb=w_l, peak_mb=peak_l, tok_s=tok_s_l)
show_ledger()

trainable: 589,824 / 125,029,632 = 0.472%  (expect 589,824 / 0.472%)
with adapters — general PPL 29.03 (expect 29.03) | held-out resp PPL 20.96 (expect 20.96)
✓ 48 adapters installed, base frozen, identity verified at model scale

[lora] step 100 | loss 2.715 | held-out resp PPL 14.61 | 17s
[lora] step 200 | loss 2.715 | held-out resp PPL 14.30 | 34s
[lora] step 300 | loss 2.734 | held-out resp PPL 14.27 | 51s

LoRA SFT: held-out resp PPL 20.96 -> 14.27 (NB11: 14.27) | general PPL 29.03 -> 29.07 (NB11: 29.06)

=== Behavioral suite, LoRA SFT (compare NB11 byte-for-byte) ===

[   one_word] FAIL | stopped: True | 'France is the capital of France.'
[three_lines] FAIL | stopped: True | 'The following animals are listed in order of their number:\n\nAsteroid\n\nAsteroid-like creatu'
[  uppercase] FAIL | stopped: True | 'Name a color.'
[     yes_no] FAIL | stopped: True | 'The sun is a star.'
[    count_5] FAIL | stopped: True | '1) Count from 1 to 5, separated by commas.\n2) Count from 1 to 5

In [ ]:
# Cell 9 — Stage 3: merge (fold BA into W), unwrap, measure recovery
#
# NB10 Cell 8 machinery, pipeline-adapted: detach gate kept (proof the
# base never moved), unmerge dropped (demonstrated in NB10; pipeline is
# one-way), and one NEW step — structural UNWRAP. A scale-zeroed branch
# still launches kernels (the 61.1 tok/s row is the proof); zero overhead
# requires removing the wrappers from the module tree entirely.

assert not globals().get("_stage3_merged", False), \
    "Stage 3 already committed — re-run Cells 7-8 to rebuild the attached state"

all_wrapped = [mod for block in model.transformer.h
                   for mod in (block.attn.c_attn, block.attn.c_proj)]
saved_scales = [[ad.scale for ad in m.adapters] for m in all_wrapped]

# --- 1) DETACH gate: zero every scale -> must read pristine on both axes ---
for m in all_wrapped:
    for ad in m.adapters:
        ad.scale = 0.0
ppl_g, _ = perplexity(model, general_ids)
ppl_r, _ = response_ppl(model, held_encoded)
print(f"DETACHED:  general {ppl_g:.2f} (expect 29.03) | resp {ppl_r:.2f} (expect 20.96)")
assert abs(ppl_g - 29.03) < 0.02 and abs(ppl_r - 20.96) < 0.02
print("✓ base never moved through 300 steps of training\n")
for m, sc in zip(all_wrapped, saved_scales):          # reattach
    for ad, s_val in zip(m.adapters, sc):
        ad.scale = s_val

# --- 2) MERGE: fold scale*(A^T B^T) into base columns (Conv1D layout) ---
adapter_bytes = sum(ad.A.numel() + ad.B.numel()
                    for m in all_wrapped for ad in m.adapters) * 4
with torch.no_grad():
    for m in all_wrapped:
        for (s, e), ad in zip(m.slices, m.adapters):
            m.base.weight[:, s:e] += ad.scale * (ad.A.T @ ad.B.T)

# --- 3) UNWRAP: physically remove the LoRAConv1D wrappers ---
for block in model.transformer.h:
    block.attn.c_attn = block.attn.c_attn.base
    block.attn.c_proj = block.attn.c_proj.base
_stage3_merged = True
del all_wrapped, saved_scales
gc.collect(); torch.cuda.empty_cache()

assert not any(isinstance(m, LoRAConv1D) for m in model.modules()), "wrappers remain"
n_total = sum(p.numel() for p in model.parameters())
assert n_total == 124_439_808, "parameter count != bare GPT-2 after unwrap"
print(f"Unwrapped: {n_total:,} params, module tree is bare GPT-2 again")
print(f"(adapter payload folded in: {adapter_bytes/1e6:.2f} MB — "
      f"1 : {497.76e6/adapter_bytes:.0f} against the base)\n")

# --- 4) Equality gates: merged model == attached model, both axes ---
ppl_g_m, _ = perplexity(model, general_ids)
ppl_r_m, _ = response_ppl(model, held_encoded)
print(f"MERGED:    general {ppl_g_m:.2f} (attached: {ppl_gen_l:.2f}) | "
      f"resp {ppl_r_m:.2f} (attached: {ppl_resp_l:.2f})")
assert abs(ppl_g_m - ppl_gen_l) < 0.02 and abs(ppl_r_m - ppl_resp_l) < 0.02

suite_merged = run_suite(model, verbose=False)
n_div = sum(1 for k in suite_lora if suite_merged[k][0] != suite_lora[k][0])
nA, nS = suite_scores(suite_merged)
print(f"Suite: Tier A {nA}/6 | stop {nS}/10 | diverged from attached: {n_div}/10\n")

# --- 5) The recovery test: cost row ---
w_m  = weights_mb(model)
tok_s_m = gen_speed(model)
peak_m  = peak_infer_mb(model)
record("merged_fp32",
       general=float(f"{ppl_g_m:.2f}"), resp=float(f"{ppl_r_m:.2f}"),
       tierA=f"{nA}/6", stop=f"{nS}/10",
       weights_mb=w_m, peak_mb=peak_m, tok_s=tok_s_m)
print(f"\nRecovery: attached {tok_s_l:.1f} tok/s -> merged {tok_s_m:.1f} tok/s "
      f"(pristine fp32 was {tok_s_32:.1f})")
show_ledger()

DETACHED:  general 29.03 (expect 29.03) | resp 20.96 (expect 20.96)
✓ base never moved through 300 steps of training

Unwrapped: 124,439,808 params, module tree is bare GPT-2 again
(adapter payload folded in: 2.36 MB — 1 : 211 against the base)

MERGED:    general 29.07 (attached: 29.07) | resp 14.27 (attached: 14.27)

Tier A: 0/6 checks passed | EOS self-stop: 10/10 prompts
Suite: Tier A 0/6 | stop 10/10 | diverged from attached: 0/10

[ledger] merged_fp32: general=29.07 | resp=14.27 | tierA=0/6 | stop=10/10 | weights_mb=497.76 | peak_mb=529.69 | tok_s=110.73

Recovery: attached 61.1 tok/s -> merged 110.7 tok/s (pristine fp32 was 87.7)
           stage |     general |        resp |       tierA |        stop |  weights_mb |     peak_mb |       tok_s |
--------------------------------------------------------------------------------------------------------------------
   pristine_fp32 |       29.03 |       20.96 |         0/6 |        0/10 |      497.76 |      520.42 |       87.69 |
   p

In [ ]:
# Cell 9b — Anomaly diagnostic: is the cost harness seam-identical?
#
# merged_fp32 (110.7) > pristine_fp32 (87.7) is architecturally impossible —
# identical module tree, dtype, param count. Hypothesis: gen_speed is
# sensitive to SESSION STATE (allocator pools, autotune caches, clocks),
# so cross-cell tok/s comparisons are contaminated. Test: both models,
# back-to-back, interleaved, right now.

pristine_probe = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
pristine_probe.eval()

# Interleaved A/B — alternating order cancels any residual drift
p_runs, m_runs = [], []
for _ in range(3):
    p_runs.append(gen_speed(pristine_probe, n_runs=1))
    m_runs.append(gen_speed(model, n_runs=1))
p_now, m_now = float(np.median(p_runs)), float(np.median(m_runs))

print(f"pristine fp32, NOW: {p_now:.1f} tok/s   (Cell 6 measured: 87.7)")
print(f"merged   fp32, NOW: {m_now:.1f} tok/s   (Cell 9 measured: 110.7)")
print(f"pristine-vs-merged gap under identical session state: "
      f"{100*abs(m_now-p_now)/p_now:.1f}%")

del pristine_probe; gc.collect(); torch.cuda.empty_cache()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

pristine fp32, NOW: 105.6 tok/s   (Cell 6 measured: 87.7)
merged   fp32, NOW: 108.8 tok/s   (Cell 9 measured: 110.7)
pristine-vs-merged gap under identical session state: 3.1%


In [ ]:
# Cell 10 — Stage 4a: save the merged checkpoint; fp16 deployment cast
#
# The merged fp32 model IS the pipeline artifact — checkpoint it before
# any cast destroys it. NF4 (Cell 11) quantizes at load time from disk,
# mirroring real deployment: one master checkpoint, N serving formats.
# tok_s recorded but provisional until the Stage 5 canonical benchmark.

CKPT = "/content/gpt2_dolly_merged"
model.save_pretrained(CKPT)
tokenizer.save_pretrained(CKPT)
import os
ckpt_mb = sum(os.path.getsize(os.path.join(CKPT, f))
              for f in os.listdir(CKPT)) / 1e6
print(f"Merged checkpoint saved: {CKPT}  ({ckpt_mb:.1f} MB on disk)\n")

# ---- deployment cast: fp16 ----
model = model.half()
gc.collect(); torch.cuda.empty_cache()

ppl_g_16, _ = perplexity(model, general_ids)
ppl_r_16, _ = response_ppl(model, held_encoded)
print(f"merged fp16: general {ppl_g_16:.2f} (fp32: 29.07) | "
      f"resp {ppl_r_16:.2f} (fp32: 14.27)")

suite_m16 = run_suite(model, verbose=False)
n_div = sum(1 for k in suite_merged if suite_m16[k][0] != suite_merged[k][0])
nA, nS = suite_scores(suite_m16)
print(f"Suite: Tier A {nA}/6 | stop {nS}/10 | diverged from merged fp32: {n_div}/10")
if n_div:
    for k in suite_merged:
        if suite_m16[k][0] != suite_merged[k][0]:
            print(f"  [{k}] fp32: {suite_merged[k][0][:70]!r}")
            print(f"  [{k}] fp16: {suite_m16[k][0][:70]!r}")

w_16  = weights_mb(model)
tok_s_16m = gen_speed(model)
peak_16m  = peak_infer_mb(model)
record("merged_fp16",
       general=float(f"{ppl_g_16:.2f}"), resp=float(f"{ppl_r_16:.2f}"),
       tierA=f"{nA}/6", stop=f"{nS}/10",
       weights_mb=w_16, peak_mb=peak_16m, tok_s=tok_s_16m)
show_ledger()

# fp16 merged model stays live — Cell 11 loads the NF4 copy alongside it,
# then releases this one after the A/B.

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged checkpoint saved: /content/gpt2_dolly_merged  (501.3 MB on disk)

merged fp16: general 29.06 (fp32: 29.07) | resp 14.27 (fp32: 14.27)

Tier A: 0/6 checks passed | EOS self-stop: 10/10 prompts
Suite: Tier A 0/6 | stop 10/10 | diverged from merged fp32: 2/10
  [three_lines] fp32: 'The following animals are listed in order of their number:\n\nAsteroid\n\n'
  [three_lines] fp16: 'The first is a dog, the second is a cat, and the third is a dog.'
  [brainstm] fp32: 'I would like to give a birthday gift to a chess enthusiast.\nI would li'
  [brainstm] fp16: 'A birthday gift is a gift that you give to someone who is interested i'
[ledger] merged_fp16: general=29.06 | resp=14.27 | tierA=0/6 | stop=10/10 | weights_mb=248.88 | peak_mb=285.52 | tok_s=107.51
           stage |     general |        resp |       tierA |        stop |  weights_mb |     peak_mb |       tok_s |
--------------------------------------------------------------------------------------------------------------------
   

In [ ]:
# Cell 11 — Stage 4b: NF4 quantization of the merged checkpoint
#
# Deployment mirror: fp32 master on disk -> 4-bit serving format at load.
# NB10 lessons applied: torch_dtype=fp16 (else embeddings/LNs stay fp32),
# device_map placement (no .to() on 4-bit). NB10 asked the TRAINING
# question (quantize base, adapt on top); this is the SERVING question
# (quantize the finished product). weights_mb cross-checked against
# allocator delta — packed 4-bit storage can fool parameter accounting.

try:
    import bitsandbytes as bnb
except ImportError:
    import subprocess; subprocess.run(["pip", "install", "-q", "bitsandbytes"])
    import bitsandbytes as bnb
from transformers import BitsAndBytesConfig

gc.collect(); torch.cuda.empty_cache()
mem_before = torch.cuda.memory_allocated() / 1e6

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
model_nf4 = GPT2LMHeadModel.from_pretrained(
    CKPT, quantization_config=bnb_cfg,
    torch_dtype=torch.float16, device_map={"": 0})
model_nf4.eval()

mem_after = torch.cuda.memory_allocated() / 1e6
w_nf4_params = sum(p.numel() * p.element_size() for p in model_nf4.parameters()) / 1e6
w_nf4_alloc  = mem_after - mem_before
print(f"NF4 model loaded from {CKPT}")
print(f"weights via parameter accounting: {w_nf4_params:.2f} MB")
print(f"weights via allocator delta:      {w_nf4_alloc:.2f} MB  <- the honest number\n")

# ---- quality axes, same three harnesses ----
ppl_g_q, _ = perplexity(model_nf4, general_ids)
ppl_r_q, _ = response_ppl(model_nf4, held_encoded)
print(f"NF4: general {ppl_g_q:.2f} (fp16: 29.06, NB10 pristine-NF4: 31.20) | "
      f"resp {ppl_r_q:.2f} (fp16: 14.27)")
print(f"Tax: general +{ppl_g_q-29.06:.2f} | resp +{ppl_r_q-14.27:.2f}\n")

print("=== Behavioral suite, NF4 ===\n")
suite_nf4 = run_suite(model_nf4)
n_div = sum(1 for k in suite_m16 if suite_nf4[k][0] != suite_m16[k][0])
print(f"Diverged from merged fp16: {n_div}/10")

# ---- cost row (tok_s provisional as ever) ----
nA, nS = suite_scores(suite_nf4)
tok_s_q = gen_speed(model_nf4)
peak_q  = peak_infer_mb(model_nf4)
record("merged_nf4",
       general=float(f"{ppl_g_q:.2f}"), resp=float(f"{ppl_r_q:.2f}"),
       tierA=f"{nA}/6", stop=f"{nS}/10",
       weights_mb=w_nf4_alloc, peak_mb=peak_q, tok_s=tok_s_q)
show_ledger()
# Both serving formats stay live: Stage 5 benchmarks them back-to-back.

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


NF4 model loaded from /content/gpt2_dolly_merged
weights via parameter accounting: 121.48 MB
weights via allocator delta:      125.30 MB  <- the honest number

NF4: general 31.25 (fp16: 29.06, NB10 pristine-NF4: 31.20) | resp 15.05 (fp16: 14.27)
Tax: general +2.19 | resp +0.78

=== Behavioral suite, NF4 ===

[   one_word] FAIL | stopped: True | 'France is the capital of France.'


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[three_lines] FAIL | stopped: False | 'The following animals are listed in alphabetical order:\n\nA cat, a dog, a cat, a dog, a cat'
[  uppercase] FAIL | stopped: False | 'Color: Black\n\nColor: White\n\nColor: Red\n\nColor: Green\n\nColor: Blue\n\nColor: Red\n\nColor: Gree'
[     yes_no] FAIL | stopped: True | 'The sun is a star.'
[    count_5] FAIL | stopped: False | '1) 1) 1) 1) 1) 1) 1) 1) 1) 1) 1) 1) 1) 1) 1) 1) 1) 1) 1) 1) 1) 1) 1) 1) 1) 1) 1) 1) 1) 1) '
[ctx_extract] FAIL | stopped: True | 'The Apollo 11 mission landed the first humans on the Moon in 1969.'
[    explain]   | stopped: True | 'The sky is blue in two sentences.'
[   brainstm]   | stopped: False | "I'm a chess enthusiast. I love to play chess. I love to play chess. I love to play chess. "
[    factual]   | stopped: True | 'The novel Moby-Dick was published in 1885.'
[     refuse]   | stopped: True | 'The library opens at nine.'

Tier A: 0/6 checks passed | EOS self-stop: 6/10 prompts
Diverged from merged fp16: 4/10


In [ ]:
# Cell 12 — Stage 5: canonical latency matrix — one session seam
#
# The 9b lesson applied: every configuration benchmarked back-to-back,
# interleaved-warm, in ONE cell. This column replaces the provisional
# tok_s entries. Grid: format {fp32, fp16, nf4} x cache {on, off} x
# length {32, 128, 512}. Formats are properties of the ARTIFACT; pristine
# rows inherit their format's speed (9b: architecture identity, 3.1% gap).

model_fp32 = GPT2LMHeadModel.from_pretrained(CKPT).to(device)
model_fp32.eval()
FORMATS = [("fp32", model_fp32), ("fp16", model), ("nf4", model_nf4)]

for _, m in FORMATS:                       # interleaved warmup, all formats
    gen_speed(m, n_new=32, n_runs=1)

LENGTHS = [32, 128, 512]
matrix = {}
print(f"{'format':>8} | {'cache':>6} |" + "".join(f"{n:>10} |" for n in LENGTHS))
print("-" * 50)
for fname, m in FORMATS:
    for cache in [True, False]:
        row = []
        for n_new in LENGTHS:
            r = gen_speed(m, n_new=n_new, n_runs=3, use_cache=cache)
            matrix[(fname, cache, n_new)] = r
            row.append(r)
        print(f"{fname:>8} | {str(cache):>6} |" +
              "".join(f"{v:>10.1f} |" for v in row))

print("\nCache value by length (cached/uncached):")
for fname, _ in FORMATS:
    ratios = [matrix[(fname, True, n)] / matrix[(fname, False, n)] for n in LENGTHS]
    print(f"  {fname}: " + " | ".join(f"{n}tok {r:.2f}x" for n, r in zip(LENGTHS, ratios)))

# ---- canonical tok_s -> ledger (cached, 128 = the benchmark condition) ----
c32, c16, cq = (matrix[(f, True, 128)] for f in ["fp32", "fp16", "nf4"])
for stage, v in [("pristine_fp32", c32), ("merged_fp32", c32),
                 ("pristine_fp16", c16), ("merged_fp16", c16),
                 ("merged_nf4", cq)]:
    LEDGER[stage]["tok_s"] = v
LEDGER["lora_sft_fp32"]["tok_s"] = None   # config gone; direction-only claim, see summary

# ---- repair the contaminated NF4 peak: measure SOLO ----
del model_fp32, model
for _n in ["FORMATS"]: del globals()[_n]
gc.collect(); torch.cuda.empty_cache()
peak_q_solo = peak_infer_mb(model_nf4)
LEDGER["merged_nf4"]["peak_mb"] = peak_q_solo
print(f"\nNF4 peak, solo (repaired): {peak_q_solo:.2f} MB "
      f"(contaminated reading was 410.82 — fp16 model was co-resident)")

print("\n=== THE DEPLOYMENT LEDGER — canonical ===")
show_ledger()
model = model_nf4   # the surviving serving artifact, for any later cell

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

  format |  cache |        32 |       128 |       512 |
--------------------------------------------------
    fp32 |   True |     104.8 |      82.7 |     107.7 |
    fp32 |  False |      85.9 |     108.4 |      49.7 |
    fp16 |   True |      99.3 |      85.3 |     103.0 |
    fp16 |  False |      83.3 |     109.6 |     108.1 |
     nf4 |   True |      48.6 |      54.7 |      51.5 |
     nf4 |  False |      45.9 |      44.4 |      43.1 |

Cache value by length (cached/uncached):
  fp32: 32tok 1.22x | 128tok 0.76x | 512tok 2.17x
  fp16: 32tok 1.19x | 128tok 0.78x | 512tok 0.95x
  nf4: 32tok 1.06x | 128tok 1.23x | 512tok 1.20x

NF4 peak, solo (repaired): 165.46 MB (contaminated reading was 410.82 — fp16 model was co-resident)

=== THE DEPLOYMENT LEDGER — canonical ===
           stage |     general |        resp |       tierA |        stop |  weights_mb |     peak_mb |       tok_s |
---------------------------------------------------------------------------------------------------------

In [ ]:
# Cell 12b — Noise-floor diagnostic: the incoherent matrix cells, interleaved
#
# fp32 cached-128 (82.7) < uncached-128 (108.4) is physically impossible on
# identical weights — cached does strictly less work. Hypothesis: within-grid
# clock/thermal drift; in the overhead-bound regime run-to-run noise ~ the
# effects being read. Test: the two contradicting cells, strictly
# alternating, 7 runs each, full spread reported.

model_fp32 = GPT2LMHeadModel.from_pretrained(CKPT).to(device)
model_fp32.eval()
gen_speed(model_fp32, n_new=128, n_runs=1)   # warmup

runs = {True: [], False: []}
for _ in range(7):
    for cache in [True, False]:              # strict alternation
        runs[cache].append(gen_speed(model_fp32, n_new=128, n_runs=1, use_cache=cache))

for cache in [True, False]:
    v = np.array(runs[cache])
    print(f"fp32 128tok cache={str(cache):>5}: "
          f"min {v.min():.1f} | median {np.median(v):.1f} | max {v.max():.1f} "
          f"| spread {100*(v.max()-v.min())/np.median(v):.0f}%")
print(f"\ncache ratio, fair seam: "
      f"{np.median(runs[True])/np.median(runs[False]):.2f}x  "
      f"(grid read 0.76x)")

del model_fp32; gc.collect(); torch.cuda.empty_cache()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

fp32 128tok cache= True: min 37.8 | median 73.3 | max 108.5 | spread 96%
fp32 128tok cache=False: min 57.5 | median 100.1 | max 107.6 | spread 50%

cache ratio, fair seam: 0.73x  (grid read 0.76x)


In [ ]:
# Cell 12c — Mechanism probe: raw forward costs, no generate() machinery
#
# Question: is a cached step (1-token forward + past) actually cheaper than
# an uncached step (full-sequence forward) at our lengths, once measured as
# BARE model calls? If a full forward at L~300 costs about the same as a
# 1-token cached step, the fixed floor dominates both and the cache can't
# win until L is large — mechanism confirmed independent of HF's loop.

model_fp32 = GPT2LMHeadModel.from_pretrained(CKPT).to(device)
model_fp32.eval()

@torch.no_grad()
def time_forward(m, L, n=20):
    """Median ms for one full forward at sequence length L."""
    x = torch.randint(0, 50257, (1, L), device=device)
    m(x)                                        # warmup
    ts = []
    for _ in range(n):
        torch.cuda.synchronize(); t0 = time.time()
        m(x)
        torch.cuda.synchronize(); ts.append((time.time() - t0) * 1e3)
    return float(np.median(ts))

@torch.no_grad()
def time_cached_step(m, L_past, n=20):
    """Median ms for one 1-token forward with a past of length L_past."""
    ctx = torch.randint(0, 50257, (1, L_past), device=device)
    past = m(ctx, use_cache=True).past_key_values
    tok = torch.randint(0, 50257, (1, 1), device=device)
    m(tok, past_key_values=past)                # warmup
    ts = []
    for _ in range(n):
        torch.cuda.synchronize(); t0 = time.time()
        m(tok, past_key_values=past)
        torch.cuda.synchronize(); ts.append((time.time() - t0) * 1e3)
    return float(np.median(ts))

print(f"{'L':>6} | {'full fwd (ms)':>14} | {'cached step (ms)':>17} | {'full/cached':>12}")
print("-" * 60)
for L in [50, 150, 300, 600]:
    f = time_forward(model_fp32, L)
    c = time_cached_step(model_fp32, L)
    print(f"{L:>6} | {f:>14.2f} | {c:>17.2f} | {f/c:>12.2f}")

del model_fp32; gc.collect(); torch.cuda.empty_cache()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

     L |  full fwd (ms) |  cached step (ms) |  full/cached
------------------------------------------------------------
    50 |          19.56 |             13.67 |         1.43
   150 |          15.38 |             11.97 |         1.29
   300 |          25.83 |             11.99 |         2.15
   600 |          50.22 |              8.65 |         5.81


## Cell 13 — The arc in retrospect: 21 days as one argument

**The argument.** A language model is a next-token predictor; everything else
is arrangement. Days 1–10 built the predictor (attention → blocks → positions
→ a 44.8M GPT → training → decoding) and learned its failure modes on a
146-token corpus. Days 11–14 made it modern and cheap (RoPE/GQA/SwiGLU;
fp16, checkpointing, accumulation, KV-cache). Days 15–20 stopped building
and started *steering*: full FT and what it forgets (NB09), adapters and
what they guarantee (NB10), instruction data and what arrangement alone can
teach (NB11). Day 21 assembled the pipeline and priced it.

**The ledger, read as one sentence.** SFT moved response PPL 20.96→14.27 and
behavioral stop 0/10→10/10 at +0.04 general PPL and +2.36 MB; merging made
the adapters free; fp16 halved memory for nothing at all (0.00/−0.01 PPL);
NF4 halved it again for +2.19/+0.78 PPL, 6/10 stop, and ~2× latency; and no
transformation anywhere in the pipeline moved Tier A off 0/6 — form is
cheap, capability is not for sale at 124M.

**The lenses that survived all 21 days:**

1. *Variance control.* From the std=0.02 bug (loss 306.9→10.95, Day 8) to
   LoRA's zero-init B (identity at init, staggered wake-up, dL/dA ∝ Bᵀ=0)
   — initialization is a statement about signal power, and every init in
   the curriculum was eventually read that way.
2. *Measurement discipline.* One perplexity harness, four notebooks, 29.03
   unbroken; a full experiment (NB11's LoRA SFT) re-run in a different file
   reproducing to the hundredth (14.27/29.07 vs 14.27/29.06). And its Day-21
   extension, learned the hard way: **the rule covers time**. PPL harnesses
   are seam-identical because the code pins the seam; latency harnesses are
   not, because session state (allocator, autotune, clocks) is part of the
   harness and it drifts. Cells 9b/12b measured the drift at up to 96%
   spread; every speed claim in this notebook is coarse-structure only.
3. *Fixed costs dominate at toy scale* — the master lesson, five sightings:
   optimizer state (NB08), vocab projection (NB08), eval padding (NB11),
   and twice today — fp16 buying ~0 latency (overhead-bound decoding), and
   the KV-cache *inverting* below a ~250-token break-even because a full
   forward at L=150 costs only 1.29× a single cached step. Small models
   don't just shrink the effects; they reorder which term is binding.
4. *Form/content decoupling.* The wrong Moby-Dick year (1885) survived four
   independently trained variants and two quantization formats. SFT taught
   answering; nothing taught knowing. The gap between 10/10 stop and 0/6
   compliance is the size of the problem RLHF and scale exist to address.

**What this curriculum did not teach, honestly:** RLHF/preference learning
(the suite's confident-fluent-wrong outputs are its motivation, unbuilt);
scale (every "at toy scale" caveat marks a conclusion that may invert at
7B, exactly as the cache inverted downward); data engineering beyond one
clean 15k-example set; and interpretability — though the artifacts are
sitting here as specimens: the ΔW spectra (which directions did SFT actually
use?), the template-keying behavior (what circuit fires on `### Response:`?),
and the 1885 error (where does a wrong fact live, that four trainings and
NF4 couldn't dislodge it?). Those are the natural next threads.

In [ ]:
# Cell 14 — Closing summary: NB12, Day 21, and the curriculum complete
print("""
==========================================================================
 NOTEBOOK 12 — INTEGRATION: THE FULL PIPELINE (Day 21)
 pretrained -> SFT -> merge -> quantize -> serve, one yardstick throughout
==========================================================================

THE DEPLOYMENT LEDGER (canonical; tok_s = cached-128, coarse-structure only)

           stage | general |  resp | tierA |  stop | weights |  peak | tok/s
 ---------------------------------------------------------------------------
   pristine fp32 |   29.03 | 20.96 |   0/6 |  0/10 |   497.8 | 520.4 |  ~83
   pristine fp16 |   29.03 | 20.96 |   0/6 |  0/10 |   248.9 | 277.0 |  ~85
   LoRA SFT fp32 |   29.07 | 14.27 |   0/6 | 10/10 |   500.1 | 534.3 |    -
     merged fp32 |   29.07 | 14.27 |   0/6 | 10/10 |   497.8 | 529.7 |  ~83
     merged fp16 |   29.06 | 14.27 |   0/6 | 10/10 |   248.9 | 285.5 |  ~85
      merged nf4 |   31.25 | 15.05 |   0/6 |  6/10 |   125.3 | 165.5 |  ~55

KEY FINDINGS

1. THE REPRODUCTION TEST, PASSED: NB11's LoRA SFT, re-run end-to-end in a
   new notebook from a transplanted yardstick, landed 14.27 / 29.07 vs
   NB11's 14.27 / 29.06 — to the hundredth, with byte-identical suite
   generations (Moby-Dick 1885 and all). Four notebooks, one harness,
   29.03 unbroken. The measurement discipline is not a ritual; it is
   what made this sentence checkable.
2. DEPLOYMENT TRANSFORMATIONS ARE NOT SYMMETRIC. fp16: memory halved,
   quality tax ZERO at our resolution (0.00 resp / -0.01 gen), 2/10
   greedy chains flipped, scores untouched. NF4: memory halved again,
   but the tax is real and skewed (+2.19 gen / +0.78 resp — 88% of the
   SFT gain survived) and the behavioral cost is the finding: 4/10
   chains reverted to degenerate repetition, stop 10/10 -> 6/10. The
   most robust thing SFT taught was the first thing quantization
   partially untaught — visible to the suite, invisible to PPL.
3. MERGE IS A MEASURED WIN, NOT A STRUCTURAL SLOGAN: attached adapters
   cost real per-token launch overhead (direction certain, magnitude
   measured-once-under-drift); folding BA into W and UNWRAPPING the
   module tree recovered format-baseline speed at bit-equivalent
   quality (0/10 suite divergence). 2.36 MB adapter payload, 1:211
   against the base — one artifact, N domains.
4. THE KV-CACHE INVERTS AT TOY SCALE. Break-even ~250-300 tokens of
   context (fp32), beyond 512 (fp16): below it, cached generation is
   ~25-30% SLOWER (grid 0.76x, fair seam 0.73x, bare-kernel mechanism
   1.29x full/cached at L=150 — three seams, one story). A full forward
   at our lengths is nearly free next to the fixed floor, while the
   cached path pays 24 concats+allocs per token. NB08's 1.74x remains
   true in NB08's regime; Day 21 found the regime boundary.
5. LATENCY IS THE SNEAKIEST AXIS: the harness CODE was seam-identical,
   the harness STATE was not. Cross-cell tok/s drifted 26% (Cell 9b);
   within-cell spread hit 96% on a shared T4 (Cell 12b). Every speed
   number in this notebook is coarse-structure; every PPL is exact.
   The NB08 Cell 7b lesson, latency edition, learned twice.
6. NOTHING MOVED TIER A OFF 0/6. Not SFT, not precision, not
   quantization. Form (seam, register, stop) transformed freely
   through the pipeline; capability was never in the pipeline at all.

PREDICTION-MISS LEDGER (Day 21's honest accounting)

- fp32 decode speed: predicted ~25 tok/s, measured 87.7 — 3.5x miss;
  bandwidth-bound intuition, overhead-bound reality. Worst of 21 days.
- "Merged > pristine is physically impossible": WRONG twice — first the
  session-state confound (9b), then cached-slower-than-uncached, which
  I called impossible and 12b/12c confirmed as real mechanism.
- Noise floor: predicted 10-25%, measured 50-96% — 4x miss.
- NF4 suite divergence: predicted 8-10/10, measured 4/10 — margin
  blindness, the SAME mechanism error as Cell 6, mispriced twice.
- NF4 response tax: predicted +1.5-4.0, measured +0.78 — the context-
  position effect (NB11 finding 1) reappearing on a new axis, unpriced.
- Hits, for balance: every deterministic gate (589,824; 159.8717;
  fingerprint; 29.03 x4), the reproduction test, the fp16 zero-tax
  call, NF4 general tax +2.19 vs NB10's +2.17, NF4 memory, merge
  equivalence, fp32~fp16 cached, and the revised cache ratios.

Progress: 21/21 days (100%). The curriculum is complete.
Artifact: /content/gpt2_dolly_merged — one fp32 master, three priced
serving formats, and a yardstick that followed it the whole way down.
""")


 NOTEBOOK 12 — INTEGRATION: THE FULL PIPELINE (Day 21)
 pretrained -> SFT -> merge -> quantize -> serve, one yardstick throughout

THE DEPLOYMENT LEDGER (canonical; tok_s = cached-128, coarse-structure only)

           stage | general |  resp | tierA |  stop | weights |  peak | tok/s
 ---------------------------------------------------------------------------
   pristine fp32 |   29.03 | 20.96 |   0/6 |  0/10 |   497.8 | 520.4 |  ~83
   pristine fp16 |   29.03 | 20.96 |   0/6 |  0/10 |   248.9 | 277.0 |  ~85
   LoRA SFT fp32 |   29.07 | 14.27 |   0/6 | 10/10 |   500.1 | 534.3 |    -
     merged fp32 |   29.07 | 14.27 |   0/6 | 10/10 |   497.8 | 529.7 |  ~83
     merged fp16 |   29.06 | 14.27 |   0/6 | 10/10 |   248.9 | 285.5 |  ~85
      merged nf4 |   31.25 | 15.05 |   0/6 |  6/10 |   125.3 | 165.5 |  ~55

KEY FINDINGS

1. THE REPRODUCTION TEST, PASSED: NB11's LoRA SFT, re-run end-to-end in a
   new notebook from a transplanted yardstick, landed 14.27 / 29.07 vs
   NB11's 14.27 / 29